# Using EcoFOCIpy to process raw field data

## DY2509 (EBS- Dyson)

## CTD / BTL Data

Basic workflow for each instrument grouping is *(initial archive level)*:
- SBE workflow must happen first
- Parse data from btl files into pandas dataframe

Convert to xarray dataframe for all following work *(working or final data level):
- Add metadata from cruise yaml files and/or header info
- ingest metadata from cruise / cast logs
- process data beyond simple file translate
- apply any calibrations or corrections
    + field corrections
    + offsets
    + instrument compensations
    + some QC were available... this would be old-school simple bounds mostly
- adjust time bounds and sample frequency (xarray dataframe)
- save as CF netcdf via xarray: so many of the steps above are optional
    + **ERDDAP NRT** if no corrections, offsets or time bounds are applied but some meta data is
    + **Working and awaiting QC** has no ERDDAP representation and is a holding spot
    + **ERDDAP Final** fully calibrated, qc'd and populated with meta information

Plot for preview and QC
- preview images (indiv and/or collectively)
- manual qc process
- automated qc process ML/AI

Further refinenments for ERDDAP hosting:

## Differences from EPIC standard
- previously btl files had coordinates of lat/lon/time/depth...with a bottle position / fireing order being a variable, this means that if multiple bottles are fired at the same depth, the file was not uniquely indexed and the bottle variable had to be conflated for multiple fireings
- in new version, we will index based on lat/lon/time/bottle_num (bottle number is a sequential unique value... often representing the position on the rosette), merging with CTD downcast data will require maintining a pressure/depth variable in the bottle data that can be rounded to the nearest 1m bin.  This does not solve the problem with multiple discrete samples taken at a single depth and single niskin though.

## Example below is for SBE 9/11+ V2 but the workflow is similar for any SBE instruments.

Future processing of this instrument can be a simplified (no markdown) process which can be archived so that the procedure can be traced or updated

We process each cast as an individual file so this example will not loop over all profiles.  See `example/all_casts.py` example for processing an entire cruise at once.

Adding Discrete samples such as Oxygen, Chlorophyll, Salinity to BTL Data is in `example/discrete_castdata.py`.  Its purpose is to match niskin/bottle information to depth for the discrete data.

In [1]:
import yaml
import glob

import EcoFOCIpy.io.sbe_ctd_parser as sbe_ctd_parser #<- instrument specific
import EcoFOCIpy.io.ncCFsave as ncCFsave
import EcoFOCIpy.metaconfig.load_config as load_config

The sample_data_dir should be included in the github package but may not be included in the pip install of the package

## Simple Processing - first step

In [2]:
sample_data_dir = '/Users/bell/ecoraid/2024/CTDcasts/dy2409/' #root path to cruise directory
ecofocipy_dir = '/Users/bell/Programs/EcoFOCIpy/'

In [3]:
###############################################################
# edit to point to {cruise sepcific} raw datafiles 
datafile = sample_data_dir+'rawconverted/' #<- point to cruise and process all files within
cruise_name = 'DY2409' #no hyphens
cruise_meta_file = sample_data_dir+'logs/DY2409.yaml'
inst_meta_file = sample_data_dir+'logs/FOCI_standard_CTD.yaml' #<- copy to each deployment for simplicity?
group_meta_file = ecofocipy_dir+'staticdata/institutional_meta_example.yaml'
###############################################################

#init and load data
cruise = sbe_ctd_parser.sbe_btl()
filename_list = sorted(glob.glob(datafile + '*.btl'))

cruise_data = cruise.manual_parse(filename_list)

Processing /Users/bell/ecoraid/2024/CTDcasts/dy2409/rawconverted/ctd001.btl
Processing /Users/bell/ecoraid/2024/CTDcasts/dy2409/rawconverted/ctd002.btl
Processing /Users/bell/ecoraid/2024/CTDcasts/dy2409/rawconverted/ctd003.btl
Processing /Users/bell/ecoraid/2024/CTDcasts/dy2409/rawconverted/ctd004.btl
Processing /Users/bell/ecoraid/2024/CTDcasts/dy2409/rawconverted/ctd005.btl
Processing /Users/bell/ecoraid/2024/CTDcasts/dy2409/rawconverted/ctd006.btl
Processing /Users/bell/ecoraid/2024/CTDcasts/dy2409/rawconverted/ctd007.btl
Processing /Users/bell/ecoraid/2024/CTDcasts/dy2409/rawconverted/ctd008.btl
Processing /Users/bell/ecoraid/2024/CTDcasts/dy2409/rawconverted/ctd009.btl
Processing /Users/bell/ecoraid/2024/CTDcasts/dy2409/rawconverted/ctd010.btl
Processing /Users/bell/ecoraid/2024/CTDcasts/dy2409/rawconverted/ctd011.btl
Processing /Users/bell/ecoraid/2024/CTDcasts/dy2409/rawconverted/ctd012.btl
Processing /Users/bell/ecoraid/2024/CTDcasts/dy2409/rawconverted/ctd013.btl
Processing /

In [4]:
#quick statistical look at the distribution of data for a cast
# #preview a dataframe
cruise_data['ctd001.btl'].describe()

,datetime
count,8
mean,2024-08-16 04:32:50.625000
min,2024-08-16 04:28:10
25%,2024-08-16 04:31:00.750000
50%,2024-08-16 04:33:04
75%,2024-08-16 04:34:53.250000
max,2024-08-16 04:36:57


## Create BTL report file

In [5]:
# btl report file
for cast in cruise_data.keys():
    try:
        df=cruise_data[cast]
        df['cast'] = cast.lower().split('.')[0]
        if cast.lower().split('.')[0] == 'ctd001':
            df.to_csv(f'{cruise_name}.report_btl')
        else:
            df.to_csv(f'{cruise_name}.report_btl',mode='a', header=False)
    except:
        print(f'some issue in {cast}')

## Time Properties

Not traditionally dealt with for CTD files as they are likely dynamically updated via GPS feed.  However, FOCI tends to label the date/time with the ***at depth*** time-stamp

## Depth Properties and other assumptions

- currently, all processing and binning (1m for FOCI) is done via seabird routines and the windows software.  This may change with the python ctd package for a few tasks

## Add Deployment meta information

In [6]:
#just a dictionary of dictionaries - simple
with open(cruise_meta_file) as file:
    cruise_config = yaml.full_load(file)
cruise_config[cruise_name]

{'CruiseID': 'DY2409',
 'CruiseID_Historic': '',
 'CruiseID_Alternates': '',
 'Project_Leg': '',
 'Vessel': 'NOAAS Oscar Dyson',
 'ShipID': 'DY',
 'StartDate': datetime.date(2024, 8, 16),
 'EndDate': datetime.date(2024, 8, 28),
 'Project': 'EMA SEBS',
 'ChiefScientist': 'Alex Andrews',
 'StartPort': 'Dutch Harbor, AK',
 'EndPort': 'Dutch Harbor, AK',
 'CruiseLocation': 'Bering Sea',
 'Description': 'South-East Bering Sea',
 'CruiseYear': 2024,
 'ctdlogs_pdf_name': 'DY2409-SEBS-OceanographyLogs.pdf',
 'Accession': '',
 'Links': '\r',
 'QC_Status': '',
 'QC_Signature': ''}

In [7]:
#and if you want a cast from the cruise, just use the consective cast number
cruise_config['CTDCasts']['CTD001']

{'id': 55067,
 'Vessel': 'NOAAS Oscar Dyson',
 'CruiseID': 'DY2409',
 'Project_Leg': '',
 'UniqueCruiseID': 'DY2409',
 'Project': 'SEBS',
 'StationNo_altname': '1',
 'ConsecutiveCastNo': 'CTD001',
 'LatitudeDeg': 55,
 'LatitudeMin': 30.076,
 'LongitudeDeg': 164,
 'LongitudeMin': 0.237,
 'GMTDay': 16,
 'GMTMonth': 'Aug',
 'GMTYear': 2024,
 'GMTTime': 44220,
 'DryBulb': -99.0,
 'RelativeHumidity': -999,
 'WetBulb': -99.9,
 'Pressure': -9999,
 'SeaState': '',
 'Visibility': '',
 'WindDir': -999,
 'WindSpd': -99.0,
 'CloudAmt': '',
 'CloudType': '',
 'Weather': '',
 'SurfaceTemp': -99.9,
 'BottomDepth': 94,
 'StationNameID': '1',
 'MaxDepth': 89,
 'InstrumentSerialNos': '',
 'Notes': 'Too dark for ???\r\nHit CLAMS EQ on upcast\r\n\r\nNutrient labels out of order\r\n\r\neDNA blank had issues , sink water from 2 different sinks filtering heavy brown?? will try to get DI water for  ???',
 'NutrientBtlNiskinNo': '1;2;3;4;5;6;7;8',
 'NutrientBtlNumbers': ';;34;37;46;43;31;49',
 'OxygenBtlNiskin

## Add Instrument meta information

Time, depth, lat, lon should be added regardless (always our coordinates) but for a mooring site its going to be a (1,1,1,t) dataset
The variables of interest should be read from the data file and matched to a key for naming.  That key is in the inst_config file seen below and should represent common conversion names in the raw data

In [8]:
with open(inst_meta_file) as file:
    inst_config = yaml.full_load(file)
inst_config

{'time': {'epic_key': 'TIM_601',
  'name': 'time',
  'generic_name': 'time',
  'standard_name': 'time',
  'long_name': 'date and time since reference time'},
 'depth': {'epic_key': 'D_3',
  'generic_name': 'depth',
  'units': 'meter',
  'long_name': 'depth below surface (meters)',
  'standard_name': 'depth',
  'positive': 'down'},
 'latitude': {'epic_key': 'LON_501',
  'name': 'latitude',
  'generic_name': 'latitude',
  'units': 'degrees_north',
  'long_name': 'latitude',
  'standard_name': 'latitude'},
 'longitude': {'epic_key': 'LAT_500',
  'name': 'longitude',
  'generic_name': 'longitude',
  'units': 'degrees_east',
  'long_name': 'longitude',
  'standard_name': 'longitude'},
 'temperature_ch1': {'epic_key': 'T_28',
  'generic_name': 'temp channel 1',
  'long_name': 'Sea temperature in-situ ITS-90 scale',
  'standard_name': 'sea_water_temperature',
  'units': 'degree_C'},
 'temperature_ch2': {'epic_key': 'T2_35',
  'generic_name': 'temp channel 2',
  'long_name': 'Sea temperature i

In [9]:
#sbe data uses header info to name variables... but we want standard names from the dictionary I've created, so we need to rename column variables appropriately
#rename values to appropriate names, if a value isn't in the .yaml file, you can add it

#*** biggest *** difference between moored and profile data is there may be multiple instruments with the same dataype (e.g.) temperature
# on the same platform.  We _used_ to use the phrases primary and secondary, but will now only refer to them as ch1, ch2 etc
cruise_data['ctd001.btl'] = cruise_data['ctd001.btl'].rename(columns={
                        't090c':'temperature_ch1',
                        't190c':'temperature_ch2',
                        'sal00':'salinity_ch1',
                        'sal11':'salinity_ch2',
                        'sbox0mm/kg':'oxy_conc_ch1',
                        'sbeox0ml/l':'oxy_concM_ch1',
                        'sbox1mm/kg':'oxy_conc_ch2',
                        'sbeox1ml/l':'oxy_concM_ch2',
                        'sbeox0ps':'oxy_percentsat_ch1',
                        'sbeox1ps':'oxy_percentsat_ch2',
                        'sigma-t00':'sigma_t_ch1',
                        'sigma-t11':'sigma_t_ch2',
                        'cstarat0':'Attenuation',
                        'cstartr0':'Transmittance',
                        'fleco-afl':'chlor_fluorescence',
                        'turbwetntu0':'turbidity',
                        'empty':'empty', #this will be ignored
                        'prdm':'Pressure [dbar]',
                        'flag':'flag'})

cruise_data['ctd001.btl'].sample()

,salinity_ch2,salinity_ch1,oxy_concM_ch1,oxy_percentsat_ch1,oxy_conc_ch1,oxy_concM_ch2,oxy_percentsat_ch2,oxy_conc_ch2,sigma_t_ch1,sigma_t_ch2,...,c1ms/cm,chlor_fluorescence,temperature_ch1,temperature_ch2,turbidity,par,scan,Pressure [dbar],datetime,cast
bottle,,,,,,,,,,,,,,,,,,,,,
7.0,31.8353,31.8333,6.4614,99.827,281.661,6.1163,94.496,266.619,24.5134,24.5149,...,34.807617,0.4676,9.7973,9.7975,0.1226,0.0,21315.0,10.828,2024-08-16 04:35:42,ctd001


In [10]:
cruise_data['ctd001.btl'].columns

Index(['salinity_ch2', 'salinity_ch1', 'oxy_concM_ch1', 'oxy_percentsat_ch1',
       'oxy_conc_ch1', 'oxy_concM_ch2', 'oxy_percentsat_ch2', 'oxy_conc_ch2',
       'sigma_t_ch1', 'sigma_t_ch2', 'c0ms/cm', 'c1ms/cm',
       'chlor_fluorescence', 'temperature_ch1', 'temperature_ch2', 'turbidity',
       'par', 'scan', 'Pressure [dbar]', 'datetime', 'cast'],
      dtype='str')

## Add institutional meta-information


In [11]:
with open(group_meta_file) as file:
    group_config = yaml.full_load(file)
group_config

{'source_documents': 'http://www.oceansites.org/docs/oceansites_data_format_reference_manual.pdf',
 'institution': 'Pacific Marine Environmental Lab (PMEL)',
 'project': 'EcoFOCI',
 'project_url': 'https://www.ecofoci.noaa.gov',
 'principal_investigator': 'Phyllis Stabeno',
 'principal_investigator_email': 'phyllis.stabeno (at) noaa.gov',
 'creator_name': 'Shaun Bell',
 'creator_email': 'shaun.bell (at) noaa.gov',
 'creator_institution': 'PMEL',
 'keywords': 'Mooring, Oceanographic',
 'comment': 'Provisional data',
 'sea_area': 'Bering Sea (BS)',
 'featureType': 'timeSeries',
 'conventions': '”CF-1.6, ~OceanSITES-1.5, ACDD-1.2”',
 'license': '',
 'references': '',
 'citation': '',
 'acknowledgement': ''}

In [12]:
# Add meta data and prelim processing based on meta data
# Convert to xarray and add meta information - save as CF netcdf file
# pass -> data, instmeta, depmeta
cruise_data_nc = ncCFsave.EcoFOCI_CFnc(df=cruise_data['ctd001.btl'], 
                                instrument_yaml=inst_config, 
                                operation_yaml=cruise_config,
                                operation_type='ctd')
cruise_data_nc

In [13]:
cruise_data_nc.get_xdf()

<xarray.Dataset> Size: 1kB
Dimensions:             (bottle: 8)
Coordinates:
  * bottle              (bottle) object 64B 1.0 2.0 3.0 4.0 5.0 6.0 7.0 8.0
Data variables: (12/21)
    salinity_ch2        (bottle) object 64B 32.5029 32.5116 ... 31.8353 31.8321
    salinity_ch1        (bottle) object 64B 32.5006 32.5096 ... 31.8333 31.8329
    oxy_concM_ch1       (bottle) object 64B 5.213 5.2221 ... 6.4614 6.4631
    oxy_percentsat_ch1  (bottle) object 64B 72.606 72.849 ... 99.827 99.872
    oxy_conc_ch1        (bottle) object 64B 226.981 227.378 ... 281.661 281.735
    oxy_concM_ch2       (bottle) object 64B 4.9424 4.9418 ... 6.1163 6.1131
    ...                  ...
    turbidity           (bottle) object 64B 0.2288 0.1817 ... 0.1226 0.1383
    par                 (bottle) object 64B 0.00071419 0.00046922 ... 0.12926
    scan                (bottle) object 64B 10483.0 12752.0 ... 21315.0 23128.0
    Pressure [dbar]     (bottle) object 64B 90.225 75.786 ... 10.828 3.018
    datetime            (bottle) datetime64[us] 64B 2024-08-16T04:28:10 ... 2...
    cast                (bottle) object 64B 'ctd001' 'ctd001' ... 'ctd001'

At this point, you could save your file with the `.xarray2netcdf_save()` method and have a functioning dataset.... but it would be very simple with no additional qc, meta-data, or tuned parameters for optimizing software like ferret or erddap.

In [14]:
# expand the dimensions and coordinate variables
# renames them appropriatley and prepares them for meta-filled values
cruise_data_nc.expand_dimensions(dim_names=['latitude','longitude','time'],geophys_sort=False)

In [15]:
#build list from columsn in data - if a variable isn't in the yaml file, it will be dropped from the final data fields
cruise_data_nc.variable_meta_data(variable_keys=list(cruise_data['ctd001.btl'].columns.values),drop_missing=False)
#adding dimension meta needs to come after updating the dimension values... BUG?
cruise_data_nc.dimension_meta_data(variable_keys=['time','latitude','longitude'])

The following steps can happen in just about any order and are all meta-data driven.  Therefore, they are not required to have a functioning dataset, but they are required to have a well described dataset

In [16]:
cruise_data_nc.get_xdf()

<xarray.Dataset> Size: 1kB
Dimensions:             (latitude: 1, longitude: 1, time: 1, bottle: 8)
Coordinates:
  * latitude            (latitude) float64 8B 1e+35
  * longitude           (longitude) float64 8B 1e+35
  * time                (time) float64 8B 1e+35
  * bottle              (bottle) object 64B 1.0 2.0 3.0 4.0 5.0 6.0 7.0 8.0
Data variables: (12/21)
    salinity_ch2        (latitude, longitude, time, bottle) object 64B 32.502...
    salinity_ch1        (latitude, longitude, time, bottle) object 64B 32.500...
    oxy_concM_ch1       (latitude, longitude, time, bottle) object 64B 5.213 ...
    oxy_percentsat_ch1  (latitude, longitude, time, bottle) object 64B 72.606...
    oxy_conc_ch1        (latitude, longitude, time, bottle) object 64B 226.98...
    oxy_concM_ch2       (latitude, longitude, time, bottle) object 64B 4.9424...
    ...                  ...
    turbidity           (latitude, longitude, time, bottle) object 64B 0.2288...
    par                 (latitude, longitude, time, bottle) object 64B 0.0007...
    scan                (latitude, longitude, time, bottle) object 64B 10483....
    Pressure [dbar]     (latitude, longitude, time, bottle) object 64B 90.225...
    datetime            (latitude, longitude, time, bottle) datetime64[us] 64B ...
    cast                (latitude, longitude, time, bottle) object 64B 'ctd00...

In [17]:
#add global attributes
cruise_data_nc.deployment_meta_add(conscastno='CTD001')

#add instituitonal global attributes
cruise_data_nc.institution_meta_add(group_config)

#add creation date/time - provenance data
cruise_data_nc.provinance_meta_add()

#provide intial qc status field
cruise_data_nc.qc_status(qc_status='unqcd') #<- options are unknown, excellent, probably good, mixed, unqcd

cruise_data_nc.get_xdf()

<xarray.Dataset> Size: 1kB
Dimensions:             (latitude: 1, longitude: 1, time: 1, bottle: 8)
Coordinates:
  * latitude            (latitude) float64 8B 1e+35
  * longitude           (longitude) float64 8B 1e+35
  * time                (time) float64 8B 1e+35
  * bottle              (bottle) object 64B 1.0 2.0 3.0 4.0 5.0 6.0 7.0 8.0
Data variables: (12/21)
    salinity_ch2        (latitude, longitude, time, bottle) object 64B 32.502...
    salinity_ch1        (latitude, longitude, time, bottle) object 64B 32.500...
    oxy_concM_ch1       (latitude, longitude, time, bottle) object 64B 5.213 ...
    oxy_percentsat_ch1  (latitude, longitude, time, bottle) object 64B 72.606...
    oxy_conc_ch1        (latitude, longitude, time, bottle) object 64B 226.98...
    oxy_concM_ch2       (latitude, longitude, time, bottle) object 64B 4.9424...
    ...                  ...
    turbidity           (latitude, longitude, time, bottle) object 64B 0.2288...
    par                 (latitude, longitude, time, bottle) object 64B 0.0007...
    scan                (latitude, longitude, time, bottle) object 64B 10483....
    Pressure [dbar]     (latitude, longitude, time, bottle) object 64B 90.225...
    datetime            (latitude, longitude, time, bottle) datetime64[us] 64B ...
    cast                (latitude, longitude, time, bottle) object 64B 'ctd00...
Attributes: (12/48)
    CruiseID:                      DY2409
    CruiseID_Historic:             
    CruiseID_Alternates:           
    Project_Leg:                   
    Vessel:                        NOAAS Oscar Dyson
    ShipID:                        DY
    ...                            ...
    references:                    
    citation:                      
    acknowledgement:               
    date_created:                  2026-06-29T08:32:43Z
    date_modified:                 
    QC_indicator:                  unqcd

## Save CF Netcdf files

Currently stick to netcdf3 classic... but migrating to netcdf4 (default) may be no problems for most modern purposes.  Its easy enough to pass the `format` kwargs through to the netcdf api of xarray.

In [18]:
#loop over all casts and perform tasks shown above

for cast in cruise_data.keys():
    try:
        cruise_data[cast] = cruise_data[cast].rename(columns={
                            't090c':'temperature_ch1',
                            't190c':'temperature_ch2',
                            'sal00':'salinity_ch1',
                            'sal11':'salinity_ch2',
                            'sbox0mm/kg':'oxy_conc_ch1',
                            'sbeox0ml/l':'oxy_concM_ch1',
                            'sbox1mm/kg':'oxy_conc_ch2',
                            'sbeox1ml/l':'oxy_concM_ch2',
                            'sbeox0ps':'oxy_percentsat_ch1',
                            'sbeox1ps':'oxy_percentsat_ch2',
                            'sigma-t00':'sigma_t_ch1',
                            'sigma-t11':'sigma_t_ch2',
                            'cstarat0':'Attenuation',
                            'cstartr0':'Transmittance',
                            'fleco-afl':'chlor_fluorescence',
                            'turbwetntu0':'turbidity',
                            'prdm':'pressure',
                            'empty':'empty', #this will be ignored
                            'flag':'flag'})

        cruise_data_nc = ncCFsave.EcoFOCI_CFnc(df=cruise_data[cast], 
                                    instrument_yaml=inst_config, 
                                    operation_yaml=cruise_config,
                                    operation_type='ctd')

        cruise_data_nc.expand_dimensions(dim_names=['latitude','longitude','time'],geophys_sort=False)

        cruise_data_nc.variable_meta_data(variable_keys=list(cruise_data[cast].columns.values),drop_missing=True)
        #adding dimension meta needs to come after updating the dimension values... BUG?
        cruise_data_nc.dimension_meta_data(variable_keys=['time','latitude','longitude'])
        cruise_data_nc.temporal_geospatioal_meta_data_ctd(positiveE=False,conscastno=cast.split('.')[0])

        #add global attributes
        cruise_data_nc.deployment_meta_add(conscastno=cast.split('.')[0].upper())

        #add instituitonal global attributes
        cruise_data_nc.institution_meta_add(group_config)

        #add creation date/time - provenance data
        cruise_data_nc.provinance_meta_add()

        #provide intial qc status field
        cruise_data_nc.qc_status(qc_status='unqcd') #<- options are unknown, excellent, probably good, mixed, unqcd

        cast_label = cast.lower().split('d')[-1].split('.')[0]
        cruise_data_nc.xarray2netcdf_save(xdf = cruise_data_nc.get_xdf(),
                                   filename=cruise_name+'c'+cast_label.zfill(3)+'_btl.nc',format="NETCDF3_CLASSIC")

  
    except:
        print(f'Skipping {cast}')

## Next Steps

QC of data (plot parameters with other instruments)
- be sure to updated the qc_status and the history